# Chapter 6: Precognition (Thinking Step by Step)

**Technique:** Chain of Thought (CoT)

**Contents**
* [Lesson: Reasoning Before Answering](#lesson)
* [Exercises](#exercises)
* [Example Playground](#example-playground)

In [ ]:
# Make the project root importable from any working directory, then load the helpers.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lib" / "helpers.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.helpers import get_completion, response_text, client, MODEL, MAX_TOKENS, SAMPLING_PARAMS

<a id="lesson"></a>
## Lesson: Reasoning Before Answering

Complex classification, triage, and decisions with multiple criteria benefit from having Claude reason through the problem *before* committing to a final answer. The technique is simple: ask Claude to write a short analysis inside an `<analysis>` block first, then deliver the answer after.

```python
PROMPT = """Before answering, write a short analysis of the evidence inside <analysis></analysis> tags, then give your final answer.

Classify this log line as DEBUG, INFO, WARN, or ERROR:
[2024-03-15 14:32:01] Connection pool exhausted after 30s, falling back to single connection"""
```

Why this helps:

* **Accuracy**: the model can't take back tokens once generated. By making it analyse first, you let it surface relevant context (category definitions, edge cases) before the decision locks in.
* **Transparency**: you get an audit trail, so you can inspect the analysis to see *why* Claude chose a label, not just what it chose.
* **Calibration**: if the analysis is wrong, you can tighten the instructions. The analysis block is a debugging surface, not just a response.

### Prompt level vs. native thinking

Modern Claude models also support **native thinking** at the API level: the model's internal chain of thought, returned as separate `thinking` blocks rather than as part of the answer text. On `claude-sonnet-4-6` you switch it on with the request parameter `thinking={"type": "adaptive"}`. On current models such as the course default `claude-opus-5-5` it is **always on**: the model decides how much to reason, and you tune the depth with `output_config={"effort": "low" | "medium" | "high" | ...}`. The thinking text is omitted from responses by default; request a readable summary with `thinking={"type": "adaptive", "display": "summarized"}`. That is the **model level** version.

What you are doing in this chapter is the **prompt level** version: you ask Claude to write a visible analysis inside `<analysis>` tags as plain output text. Both approaches improve accuracy on hard tasks. The prompt level version needs no extra API parameters and puts the reasoning right in the text your application already reads, which makes it the easiest starting point.

### Name the tag after the work, not the model's thoughts

Ask for an *analysis*, an *assessment*, or an *evaluation* of the input, not for Claude's "thinking" or "scratchpad". On current models, a prompt that asks Claude to write out its internal thinking in the response (for example "reason step by step inside `<thinking>` tags") is treated as an attempt to extract the model's private reasoning and is declined: the API returns `stop_reason="refusal"` with `stop_details.category == "reasoning_extraction"` and **no text at all**. The cells below show both versions. In production code, always check `stop_reason` before using a response; the course helpers print a warning whenever Claude declines.

### Example: log line severity classification

```python
PROMPT = """Classify this log line's severity. First write a short analysis of the evidence inside <analysis></analysis> tags, then state the severity on a new line.

[2024-03-15 14:32:01] Connection pool exhausted after 30s, falling back to single connection"""
```

The model might produce:

```
<analysis>
The pool is exhausted and the system degraded to single connection mode. No crash, the fallback succeeded, but service is degraded. That is a warning condition, not a hard error.
</analysis>
WARN
```

In [ ]:
PROMPT = """Classify this log line's severity. First write a short analysis of the evidence inside <analysis></analysis> tags, then state the severity (DEBUG / INFO / WARN / ERROR) on a final line with no other text.

[2024-03-15 14:32:01] Connection pool exhausted after 30s, falling back to single connection"""

print(get_completion(PROMPT))

Now the same request, but asking Claude to transcribe its *thinking*. Instead of an answer you get a refusal with no text. Checking `stop_reason` is how your code tells "Claude declined" apart from "Claude answered with an empty string".

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{
        "role": "user",
        "content": "Classify this log line's severity. Reason step by step inside <thinking></thinking> tags, then state the severity on a final line.\n"
                   "[2024-03-15 14:32:01] Connection pool exhausted after 30s, falling back to single connection",
    }],
    **SAMPLING_PARAMS,
)

print("stop_reason:", response.stop_reason)
if response.stop_reason == "refusal" and response.stop_details:
    print("category:   ", response.stop_details.category)
else:
    print(response_text(response))

For comparison, here is the **model level** version. We ask for a summarized view of Claude's native thinking and print the `thinking` and `text` blocks separately. Notice that the answer text no longer needs to contain the reasoning.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    thinking={"type": "adaptive", "display": "summarized"},
    messages=[{
        "role": "user",
        "content": "Classify this log line's severity as DEBUG, INFO, WARN, or ERROR. Reply with the label only.\n"
                   "[2024-03-15 14:32:01] Connection pool exhausted after 30s, falling back to single connection",
    }],
)

for block in response.content:
    if block.type == "thinking":
        print("THINKING (summary):", block.thinking or "(no summary returned)")
    elif block.type == "text":
        print("ANSWER:", block.text)

<a id="exercises"></a>
## Exercises

### Exercise 6.1: Triage a support ticket with visible reasoning

**Scenario**: your support team receives tickets that fall into four categories:

* **(A) Bug report**: the software does something it shouldn't, or fails to do something it should.
* **(B) Feature request**: the user wants new capability.
* **(C) Billing**: questions or problems related to payments, subscriptions, or invoices.
* **(D) Other**: anything that doesn't fit the above.

**Task**: write a prompt that presents all four categories to Claude and asks it to classify the ticket below, letting Claude write a short analysis in an `<analysis>` block before it commits to a label.

**Success criteria**: Claude's response contains the correct label `(A)`.

In [ ]:
from lib.grading import includes_any, grade

TICKET = "The app crashes with a 500 every time I upload a PNG over 5MB."
PROMPT = "[Replace this text - include the four categories and ask Claude to classify the ticket, thinking step by step first]\n" + TICKET

response = get_completion(PROMPT)

def grade_exercise(text):
    return includes_any(text, ["(A)"])

grade(response, grade_exercise(response))

In [ ]:
from hints import exercise_6_1_hint
print(exercise_6_1_hint)

### Exercise 6.2: Letter only output for an automated router

**Scenario**: the classifier from 6.1 feeds an automated ticket router. The router reads a single character from stdin (the category letter) and fails if it receives anything else. No preamble, no parentheses, no explanation: just `A`, `B`, `C`, or `D`.

**Task**: modify your prompt so Claude outputs *only* the category letter. You can instruct Claude directly ("respond with only the single letter") or enforce it with a structured output (`output_config`). Do **not** use an assistant turn prefill, which returns a 400 on current models.

**Success criteria**: `response.strip()` is exactly `"A"`.

In [ ]:
from lib.grading import equals_exact, grade

TICKET = "The app crashes with a 500 every time I upload a PNG over 5MB."
PROMPT = "[Replace this text - classify into A/B/C/D and respond with ONLY the single letter]\n" + TICKET

response = get_completion(PROMPT)

def grade_exercise(text):
    return equals_exact(text.strip(), "A")

grade(response, grade_exercise(response))

In [ ]:
from hints import exercise_6_2_hint
print(exercise_6_2_hint)

## Common mistakes, stretch challenge, and reflection

**Common mistakes**

* **Trying to prefill the assistant turn**: appending `{"role": "assistant", "content": "("}` to the messages list to force a parenthetical label returns a **400 Bad Request** on current models. Use an explicit instruction ("respond with only the label in the format (A)") or `output_config` instead.
* **Letting preamble leak into the letter only answer**: if Claude adds "The answer is A" instead of just "A", your downstream parser breaks. Be explicit: "Respond with ONLY the single letter. No punctuation, no explanation."
* **Skipping the `<analysis>` block**: if you ask Claude to classify directly without a reasoning step, it may get edge case tickets wrong. The analysis is cheap (a few hundred tokens) and reliably improves accuracy on ambiguous inputs.
* **Asking for Claude's "thinking" or "scratchpad" in the output**: on current models this is declined with `stop_reason="refusal"` (category `reasoning_extraction`) and an empty response. Ask for an analysis of the input instead, and always check `stop_reason`.

**Stretch challenge**

Extend Exercise 6.1 to return structured output: use `output_config` with a JSON Schema that has a `category` string field (the label letter) and a `reason` string field (one sentence). Verify that `json.loads(response_text(msg))["category"] == "A"`.

**Reflect**: when does a visible `<analysis>` block help versus hurt? Consider latency (extra tokens = more time), cost (you pay for thinking tokens too), and transparency (is the reasoning useful to a human reviewer or only to the model?). When would you rely on native thinking at the API level instead, and how would you tune its `effort`?

<a id="example-playground"></a>
## Example Playground

Use the cells below to experiment freely. Try different ticket types, add more categories, or switch to structured output with a `reason` field alongside the category.

In [ ]:
# Experiment: classify a batch of tickets and display each with its reasoning
TICKETS = [
    "My invoice shows a charge I didn't authorize.",
    "Would love a dark mode option in the dashboard.",
    "Getting a 403 on every API call since yesterday's deploy.",
]

SYSTEM = """You are a support ticket classifier. Categories:
(A) Bug report
(B) Feature request
(C) Billing
(D) Other

For each ticket, first write a brief analysis inside <analysis></analysis> tags, then state the label on a new line."""

for ticket in TICKETS:
    print("--- Ticket:", ticket)
    print(get_completion(ticket, SYSTEM))
    print()